# FFTW C2C F90 Sequential

Complex Multi-Dimensional DFTs 3D

    fftw_plan fftw_plan_dft_3d(int n0, int n1, int n2,
                   fftw_complex *in, fftw_complex *out,
                   int sign, unsigned flags);                          
- http://www.fftw.org/fftw3_doc/Complex-Multi_002dDimensional-DFTs.html

In [41]:
#=-----------------------------------------------------------------------=#

## In-place c2c transform, random

In [1]:
%%writefile test06.f90
program main
    use, intrinsic :: iso_c_binding
    implicit none
    include "fftw3.f03"
    integer :: i, j, k
    integer, parameter :: N = 3, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    complex(C_DOUBLE_COMPLEX) :: s
           
    ! in-place transform (note dimension reversal)
    cdata = fftw_alloc_complex(int(L * M * N, C_SIZE_T))
    call c_f_pointer(cdata, data, [L, M, N])

    !create plan for in-place forward DFT (note dimension reversal)
    plan = fftw_plan_dft_3d(N, M, L, data, data, &
                            FFTW_FORWARD, FFTW_ESTIMATE)

    !initialize data
    data = reshape([(.50,.0), (.73,.0), (.22,.0),  &
                    (.29,.0), (.65,.0), (.41,.0),  &
                    (.69,.0), (.25,.0), (.76,.0),  &
                    (.64,.0), (.60,.0), (.73,.0),  &
                    (.93,.0), (.24,.0), (.63,.0),  &
                    (.19,.0), (.73,.0), (.77,.0),  &
                    (.93,.0), (.70,.0), (.29,.0),  &
                    (.53,.0), (.34,.0), (.20,.0),  &
                    (.91,.0), (.20,.0), (.47,.0)], &
                    shape(data), order=[3, 2, 1] )
    write(*,*) 'in:'; do k = 1, size(data,1); do j = 1, size(data,2);
        write(*, "(*(:'('f5.2','f5.2')':x))") data(k,j,:);
    enddo; write(*,*); enddo;

    !compute transform (as many times as desired)
    call fftw_execute_dft(plan, data, data)

    !print result
    write(*,*) 'out:'; do k = 1, size(data,1); do j = 1, size(data,2);
        write(*,  "(*(:'('f5.2','f5.2')':x))") data(k,j,:);
    enddo; write(*,*); enddo;
    s = sum(data)
    write(*, "('S: '*(f0.4spf0.4'j':x))") s
    
    ! save the result
    open(unit = 10, file = "data.csv", action = "write", &
         access = "sequential", status = "replace", form = "formatted")
    write(10,'(*("("g0spg0"j)":","))') s
    close(unit = 10)
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
end

Writing test06.f90


In [2]:
%%bash
gfortran test06.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3 -l m -I $(pwd)/lo/include
./a.out

 in:
( 0.50, 0.00) ( 0.73, 0.00) ( 0.22, 0.00)
( 0.29, 0.00) ( 0.65, 0.00) ( 0.41, 0.00)
( 0.69, 0.00) ( 0.25, 0.00) ( 0.76, 0.00)

( 0.64, 0.00) ( 0.60, 0.00) ( 0.73, 0.00)
( 0.93, 0.00) ( 0.24, 0.00) ( 0.63, 0.00)
( 0.19, 0.00) ( 0.73, 0.00) ( 0.77, 0.00)

( 0.93, 0.00) ( 0.70, 0.00) ( 0.29, 0.00)
( 0.53, 0.00) ( 0.34, 0.00) ( 0.20, 0.00)
( 0.91, 0.00) ( 0.20, 0.00) ( 0.47, 0.00)

 out:
(14.53, 0.00) ( 1.15, 0.03) ( 1.15,-0.03)
( 0.75, 0.65) (-0.53,-1.32) ( 0.68, 0.77)
( 0.75,-0.65) ( 0.68,-0.77) (-0.53, 1.32)

(-0.52,-0.77) ( 0.01, 0.85) (-1.25, 1.51)
(-0.95, 0.45) (-1.24,-0.11) (-0.74, 1.51)
(-0.02, 0.19) ( 1.90,-0.50) ( 0.24,-0.86)

(-0.52, 0.77) (-1.25,-1.51) ( 0.01,-0.85)
(-0.02,-0.19) ( 0.24, 0.86) ( 1.90, 0.50)
(-0.95,-0.45) (-0.74,-1.51) (-1.24, 0.11)

S: 13.5000+.0000j


In [3]:
!cat data.csv

(13.500000000000000+.11102230246251565E-015j)


### Check

In [4]:
import numpy as np
a = np.array(
[[[.50+0j, .73+0j, .22+0j], [.29+0j, .65+0j, .41+0j], [.69+0j, .25+0j, .76+0j]],
[[.64+0j, .60+0j, .73+0j], [.93+0j, .24+0j, .63+0j], [.19+0j, .73+0j, .77+0j]], 
[[.93+0j, .70+0j, .29+0j], [.53+0j, .34+0j, .20+0j], [.91+0j, .20+0j, .47+0j]]])
f = np.fft.fftn(a)

In [5]:
np.set_printoptions(precision=2, suppress=True)
print(f[:2,:,:])

[[[14.53+0.j    1.15+0.03j  1.15-0.03j]
  [ 0.74+0.65j -0.53-1.32j  0.69+0.77j]
  [ 0.74-0.65j  0.69-0.77j -0.53+1.32j]]

 [[-0.52-0.77j  0.01+0.85j -1.25+1.51j]
  [-0.95+0.45j -1.24-0.11j -0.74+1.51j]
  [-0.02+0.19j  1.9 -0.5j   0.24-0.86j]]]


In [6]:
t = np.sum(f)
print('S: {:.4f}'.format(t))

S: 13.5000-0.0000j


In [7]:
u = np.loadtxt("data.csv", dtype=np.complex128)
np.allclose(t, u)

True

## 3D 12x12x12, random

In [10]:
%%writefile test07.f90
program main
    use, intrinsic :: iso_c_binding
    implicit none
    include "fftw3.f03"
    integer :: i, j, k
    integer, parameter :: L = 12, M = 12, N = 12
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    complex(C_DOUBLE_COMPLEX) :: s
    integer, allocatable, dimension(:) :: seed
    real :: r(L, M, N)
           
    ! in-place transform (note dimension reversal)
    cdata = fftw_alloc_complex(int(L * M * N, C_SIZE_T))
    call c_f_pointer(cdata, data, [L, M, N])

    ! create plan for in-place forward DFT (note dimension reversal)
    plan = fftw_plan_dft_3d(N, M, L, data, data, &
                            FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data
    seed = [.50, .73, .22, .29, .65, .41, .69, .25, 0., 0., 0., 0.]
    call random_seed(put=seed)
    call random_number(r)
    data = cmplx(r, 0)

    ! compute transform (as many times as desired)
    call fftw_execute_dft(plan, data, data)

    ! print result
    s = sum(data)
    write(*, "('S: '*(sf8.4spf8.4'j':x))") s
    
    ! save the result
    open(unit = 10, file = "data.csv", action = "write", &
         access = "sequential", status = "replace", form = "formatted")
    write(10,'(*("("g0spg0"j)":","))') s
    close(unit = 10)
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
end

Overwriting test07.f90


In [11]:
%%bash
gfortran test07.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3 -l m -I $(pwd)/lo/include
./a.out

S: 533.9813 +0.0000j


## 3D 128x128x128, function

In [79]:
%%writefile test08.f90
program main
    use, intrinsic :: iso_c_binding
    implicit none
    include "fftw3.f03"
    integer, parameter :: N= 128, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    complex(C_DOUBLE_COMPLEX) :: s
    integer :: i, j, k
    double precision :: r
           
    ! in-place transform (note dimension reversal)
    cdata = fftw_alloc_complex(int(L * M * N, C_SIZE_T))
    call c_f_pointer(cdata, data, [L, M, N])

    ! create plan for in-place forward DFT (note dimension reversal)
    plan = fftw_plan_dft_3d(N, M, L, data, data, &
                            FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data
    do k = 1, N
        do j = 1, M
            do i = 1, L
                r = ((i-1)*N**2 + (j-1)*N  + (k-1) + 1)*1E-6
                data(i, j, k) = cmplx(r, 0)
            enddo
        enddo
    enddo
    s = sum(data)
    write(*, "('D: 'sf0.2spf0.2'j')") s

    ! compute transform (as many times as desired)
    call fftw_execute_dft(plan, data, data)

    ! print result
    s = sum(data)
    write(*, "('S: 'sf0.2spf0.2'j')") s
    
    ! save the result
    open(unit = 10, file = "data.csv", action = "write", &
         access = "sequential", status = "replace", form = "formatted")
    write(10,'(*("("g0spg0"j)":","))') s
    close(unit = 10)
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
end

Overwriting test08.f90


In [80]:
%%bash
gfortran test08.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3 -l m -I $(pwd)/lo/include
./a.out

D: 2199024.30+.00j
S: 2.10+.00j


In [58]:
D: 2199022206976.0000+.0000j

In [30]:
! cat data.csv

(2097152.0000307541+.0000000000000000j)


## 3D 500x500x500

In [1]:
%%writefile test09.f90
program main
    use, intrinsic :: iso_c_binding
    implicit none
    include "fftw3.f03"
    integer, parameter :: N = 500, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    complex(C_DOUBLE_COMPLEX) :: s
    integer :: i, j, k
    double precision :: r, t1, t2
    
    call cpu_time(t1)    ! <--- time measurement
           
    ! in-place transform (note dimension reversal)
    cdata = fftw_alloc_complex(int(L * M * N, C_SIZE_T))
    call c_f_pointer(cdata, data, [L, M, N])

    ! create plan for in-place forward DFT (note dimension reversal)
    
    plan = fftw_plan_dft_3d(N, M, L, data, data, &
                            FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data    
     do k = 1, N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k-1) + 1)*1E-6
            data(i, j, k) = cmplx(r, 0)
          enddo
        enddo
     enddo

    ! compute transform (as many times as desired)    
    call fftw_execute_dft(plan, data, data)
    s = sum(data)
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
  
    call cpu_time(t2)    ! <--- time measurement
    
    ! show the result
    write(*, "('S: 'sf0.2spf0.2'j')", advance="no") s
    write(*, "('    T: 'sf0.4' s')") t2-t1  
end

Overwriting test09.f90


In [2]:
! gfortran test09.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3 -l m -I $(pwd)/lo/include

In [3]:
! ./a.out

S: 125.00+.00j    T: 9.5348 s


## Runs on an execution node

- Compile using -O3 and uses system libraries
- Using: module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu

In [1]:
%%bash
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
gfortran -O3 -o fc2cs test09.f90 \
    -L $dir/lib -l fftw3 -l m -I $dir/include

Check

In [2]:
! ./fc2cs

S: 125.00+.00j    T: 7.6655 s


Copy to /scratch

In [3]:
%%bash
dst=/scratch${PWD#"/prj"}
mkdir -p $dst
cp fc2cs $dst

## Batch script

In [11]:
%%writefile fc2cs.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu_dev  : 20 min.,  1-4  nodes, 1/1   tasks
#   cpu_small: 72 hours, 1-20 nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name fc2cs       # Job name
#SBATCH --time=00:01:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes

echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Executable config
EXEC=$PWD/fc2cs

# Start
echo '$ srun -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting fc2cs.srm


Submit batch script

In [12]:
! sbatch fc2cs.srm
! sbatch fc2cs.srm
! sbatch fc2cs.srm

Submitted batch job 862571
Submitted batch job 862572
Submitted batch job 862573


In [9]:
! squeue -n fc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            862571  cpu_small  PD  0:00     1    1
            862572  cpu_small  PD  0:00     1    1
            862573  cpu_small  PD  0:00     1    1


In [10]:
! squeue -n fc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            862571  cpu_small  PD  0:00     1    1
            862572  cpu_small  PD  0:00     1    1
            862573  cpu_small  PD  0:00     1    1


In [16]:
! cat /scratch${PWD#"/prj"}/slurm-862571.out
! cat /scratch${PWD#"/prj"}/slurm-862572.out
! cat /scratch${PWD#"/prj"}/slurm-862573.out

- Job ID: 859733
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1248
$ srun -n 1 fc2cs
-- output -----------------------------
S: 123378328.9790+.0000j    T: 9.9713 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [3]:
! sbatch --partition cpu_dev fc2cs.srm

Submitted batch job 862671


In [4]:
! squeue -n fc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID PARTITION ST       TIME  NODES
            862571 cpu_small PD       0:00      1
            862572 cpu_small PD       0:00      1
            862573 cpu_small PD       0:00      1
            862671   cpu_dev PD       0:00      1


In [1]:
! squeue -n fc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-862671.out
! cat /scratch${PWD#"/prj"}/slurm-862572.out
! cat /scratch${PWD#"/prj"}/slurm-862573.out

- Job ID: 862671
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1259
$ srun -n 1 fc2cs
-- output -----------------------------
S: 125.00+.00j    T: 9.0476 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 862572
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
$ srun -n 1 fc2cs
-- output -----------------------------
S: 125.00+.00j    T: 9.1026 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 862573
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
$ srun -n 1 fc2cs
-- output -----------------------------
S: 125.00+.00j    T: 9.1274 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### Version

In [4]:
! gfortran --version

GNU Fortran (GCC) 4.8.5 20150623 (Red Hat 4.8.5-36)
Copyright (C) 2015 Free Software Foundation, Inc.

GNU Fortran comes with NO WARRANTY, to the extent permitted by law.
You may redistribute copies of GNU Fortran
under the terms of the GNU General Public License.
For more information about these matters, see the file named COPYING



In [2]:
! module avail 2>&1 | grep -i fftw

mathlibs/fftw/3.3.8_intel
mathlibs/fftw/3.3.8_openmpi-2.0_gnu
mathlibs/fftw/3.3.8_openmpi-2.0_intel
mathlibs/fftw/3.3.8_openmpi-3.1_gnu


In [5]:
! hostnamectl | grep Operating

  Operating System: Red Hat Enterprise Linux Server 7.6 (Maipo)
